In [ ]:
!pip install -q speechbrain datasets torchaudio

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torchaudio
import speechbrain as sb
from speechbrain.dataio.dataset import DynamicItemDataset
from speechbrain.utils.Accuracy import Accuracy
from speechbrain.utils.metric_stats import MetricStats
rom speechbrain.lobes.models.huggingface_transformers.wav2vec2 import Wav2Vec2
from speechbrain.nnet.linear import Linear
from speechbrain.dataio.batch import PaddedBatch
from functools import partial
from datasets import load_dataset

In [ ]:
#hiperparâmetros
SEED = 42
BATCH_SIZE = 16
EPOCHS = 35
LR = 1e-3
NUM_CLASSES = 14
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
sb.utils.seed_everything(SEED)
hf_dataset = load_dataset("polyAI/minds14", "en-US", split="train")
split_ds = hf_dataset.train_test_split(test_size=0.2, seed=SEED)

In [ ]:
def extract_and_format(example):
    
    audio_info = example["audio"]
    array = audio_info["array"]
    
    sig = torch.tensor(array, dtype=torch.float32)
    
    sig = sig.squeeze()
    if sig.ndim == 0:
        sig = sig.unsqueeze(0)
        
    sr = audio_info["sampling_rate"]
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        sig = resampler(sig)
        
    return {
        "sig": sig,
        "target": torch.tensor([int(example["intent_class"])], dtype=torch.long)
    }

In [ ]:
@sb.utils.data_pipeline.takes("sig", "target")
@sb.utils.data_pipeline.provides("sig", "target")
def identity_pipeline(sig, target):
    yield sig
    yield target

In [ ]:
train_dict = {str(i): extract_and_format(item) for i, item in enumerate(split_ds["train"])}
valid_dict = {str(i): extract_and_format(item) for i, item in enumerate(split_ds["test"])}

In [ ]:
train_data = DynamicItemDataset(
    train_dict,
    dynamic_items=[identity_pipeline],
    output_keys=["id", "sig", "target"]
)

valid_data = DynamicItemDataset(
    valid_dict,
    dynamic_items=[identity_pipeline],
    output_keys=["id", "sig", "target"]
)

In [ ]:
class MeanPooling(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x, lengths=None):
        # x shape: [batch, time, features]
        if lengths is None:
            return x.mean(dim=1)
        # Se houver tamanhos variados, aplica a média ponderada pelas máscaras
        max_len = x.size(1)
        mask = torch.arange(max_len, device=x.device)[None, :] < (lengths * max_len)[:, None]
        mask = mask.unsqueeze(-1).float()
        return (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

In [ ]:
wav2vec2_encoder = Wav2Vec2(
    source="facebook/wav2vec2-base-960h",
    save_path="./pretrained_w2v2",
    output_norm=True,
    freeze=True
)
pooling = MeanPooling()
classifier = Linear(input_size=768, n_neurons=NUM_CLASSES)

In [ ]:
class SLUKaggleBrain(sb.Brain):
  
    def compute_forward(self, batch, stage):
        
        batch = batch.to(self.device)
        wavs, wav_lens = batch.sig

        if stage == sb.Stage.TRAIN:
            feats = self.modules.wav2vec2(wavs)
            pooled = self.hparams.pooling(feats, wav_lens)
            return self.modules.classifier(pooled).squeeze(1)


        sample_rate = 16000
        total_audio_duration_sec = (wavs.shape[1] * wav_lens).sum().item() / sample_rate

        if self.device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        feats = self.modules.wav2vec2(wavs)
        pooled = self.hparams.pooling(feats, wav_lens)
        outputs = self.modules.classifier(pooled)

        if self.device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        proc_time_sec = t1 - t0
        batch_size = wavs.shape[0]

        self.latencies_ms.append((proc_time_sec * 1000) / batch_size)
        
        
        return outputs.squeeze(1)

    def compute_objectives(self, predictions, batch, stage):
        
        targets = batch.target.squeeze(-1)
        loss = self.hparams.cost(predictions, targets)
        
        if stage != sb.Stage.TRAIN:
            preds = predictions.argmax(dim=-1)
            self.correct_preds += (preds == targets).sum().item()
            self.total_preds += targets.size(0)
        return loss

    def on_stage_start(self, stage, epoch):
        
        if stage != sb.Stage.TRAIN:
            self.correct_preds = 0
            self.total_preds = 0
            self.latencies_ms = []

    def on_stage_end(self, stage, stage_loss, epoch):
        
        if stage == sb.Stage.VALID:
            acc = (self.correct_preds / self.total_preds * 100) if self.total_preds > 0 else 0.0
            if len(self.latencies_ms) > 0:
                avg_latency = sum(self.latencies_ms) / len(self.latencies_ms)
                p95_latency = torch.quantile(torch.tensor(self.latencies_ms), 0.95).item()
            else:
                avg_latency, p95_latency = 0.0, 0.0

            
            print(f"| Época {epoch:02d} | Perda Val: {stage_loss:.4f} | Acurácia: {acc:.2f}% |")
            print(f"| Latência Média por amostra: {avg_latency:.2f} ms | P95: {p95_latency:.2f} ms |\n")

In [ ]:
modules = {
    "wav2vec2": wav2vec2_encoder,
    "classifier": classifier,
}

hparams = {
    "pooling": pooling,
    "cost": torch.nn.CrossEntropyLoss(),
    "epoch_counter": sb.utils.epoch_loop.EpochCounter(limit=EPOCHS),
}

brain = SLUKaggleBrain(
    modules=modules,
    opt_class=lambda params: torch.optim.Adam(params, lr=LR),
    hparams=hparams,
    run_opts={"device": DEVICE}
)

custom_collate = partial(PaddedBatch, padded_keys=["sig"])

In [ ]:
print("\n Iniciando Treinamento no Kaggle ")
brain.fit(
    epoch_counter=hparams["epoch_counter"],
    train_set=train_data,
    valid_set=valid_data,
    train_loader_kwargs={
        "batch_size": BATCH_SIZE,
        "collate_fn": custom_collate
    },
    valid_loader_kwargs={
        "batch_size": BATCH_SIZE,
        "collate_fn": custom_collate
    }
)